# Task 1

For the environment, I have selected `FrozenLake-v1` (4x4 Grid, Slippery). The slippery nature indicates that transitions are stochastic (moves are not always in the direction intended), making it a true Markov Decision Process rather than a simple deterministic search problem.

- State Space ($S$): 16 discrete states, representing the 4x4 grid. State 0 is the top-left (Start), and State 15 is the bottom-right (Goal).
- Action Space ($A$): 4 discrete actions:
  - 0: Move Left
  - 1: Move Down
  - 2: Move Right
  - 3: Move Up
- Reward Structure ($R$): $+1$ for reaching the Goal (State 15).
  - $0$ for stepping on frozen tiles (F), falling into a hole (H), or staying at the start (S).
- Discount Factor ($\gamma$): 0.99 for the base implementation to encourage finding a path to the goal while valuing slightly faster routes.


Policy Iteration was selected to solve the `FrozenLake-v1` MDP because it efficiently computes the optimal strategy for environments with known, discrete transition dynamics. The algorithm works by continuously alternating between policy evaluation and policy improvement. During evaluation, it calculates the expected long-term value of every state under the current navigational strategy, explicitly factoring in the transition probabilities caused by the slippery ice. During improvement, it updates the strategy by greedily choosing the action in each state that maximizes this expected discounted reward. This two-step cycle repeats iteratively until the chosen actions stop changing, guaranteeing that the algorithm has found the absolute mathematical optimal policy for reaching the goal.

In [ ]:
import numpy as np
import gymnasium as gym

def evaluate_policy(env, policy, gamma, theta=1e-8, max_eval_iters=10000):
    """Evaluates a given policy to find the state-value function."""
    V = np.zeros(env.observation_space.n)

    for i in range(max_eval_iters):
        delta = 0

        for s in range(env.observation_space.n):
            v_old = V[s]
            a = policy[s]
            # Sum over all possible transitions
            v_new = 0

            for prob, next_s, reward, terminated in env.unwrapped.P[s][a]:
                v_new += prob * (reward + gamma * V[next_s] * (not bool(terminated)))
            V[s] = v_new
            delta = max(delta, abs(v_old - V[s]))

        if delta < theta:
            # Convergence
            break
    if i == max_eval_iters - 1:
         print("Warning: Policy evaluation hit the maximum iteration limit")
    return V

def improve_policy(env, policy, V, gamma):
    """Improves the policy based on the current value function."""
    policy_stable = True

    for s in range(env.observation_space.n):
        old_action = policy[s]

        # Calculate Q-values for all actions
        action_values = np.zeros(env.action_space.n)

        for a in range(env.action_space.n):
            for prob, next_s, reward, terminated in env.unwrapped.P[s][a]:
                action_values[a] += prob * (reward + gamma * V[next_s] * (not bool(terminated)))
        # Greedily choose the best action
        best_action = np.argmax(action_values)
        policy[s] = best_action

        if old_action != best_action:
            policy_stable = False
    return policy, policy_stable

def policy_iteration(env, gamma=0.99):
    """Executes the full Policy Iteration algorithm."""
    policy = np.zeros(env.observation_space.n, dtype=int)
    is_stable = False
    iterations = 0

    while not is_stable:
        V = evaluate_policy(env, policy, gamma)
        policy, is_stable = improve_policy(env, policy, V, gamma)
        iterations += 1
    return policy, V, iterations

def print_results(policy, V, title=""):
    """Helper function to print the Value function and Policy nicely."""
    actions = {0: '←', 1: '↓', 2: '→', 3: '↑'}

    print(f"\n{title}")
    print("Value Function:")
    print(np.round(V.reshape((4, 4)), 3))
    print("\nPolicy:")
    policy_arrows = np.array([actions[a] for a in policy]).reshape((4, 4))

    # Replace terminal states with visual markers
    holes = [(1,1), (1,3), (2,3), (3,0)]

    for h in holes:
        policy_arrows[h] = 'H'
    policy_arrows[3,3] = 'G'

    print(policy_arrows)

# Run basline
env = gym.make('FrozenLake-v1', is_slippery=True)
optimal_policy, optimal_V, num_iters = policy_iteration(env, gamma=0.99)

print(f"Converged in {num_iters} iterations.")
print_results(optimal_policy, optimal_V, "Basline (Gamma = 0.99)")

Converged in 7 iterations.

Basline (Gamma = 0.99)
Value Function:
[[0.542 0.499 0.471 0.457]
 [0.558 0.    0.358 0.   ]
 [0.592 0.643 0.615 0.   ]
 [0.    0.742 0.863 0.   ]]

Policy:
[['←' '↑' '↑' '↑']
 ['←' 'H' '←' 'H']
 ['↑' '↓' '←' 'H']
 ['H' '→' '↓' 'G']]


The baseline experiment, utilizing a discount factor of gamma = 0.99, successfully converged to the optimal policy in exactly 7 iterations. The learned value function and the resulting policy map demonstrate a highly risk-averse strategy that perfectly adapts to the stochastic, slippery nature of the environment. Rather than attempting to take the shortest geometric path to the bottom-right corner, the agent learned to use the boundaries of the grid as a safety mechanism. For instance, when positioned in the top row or adjacent to holes, the policy frequently directs the agent to move directly into a wall. Due to the agent having a high probability of slipping perpendicularly to its intended direction, commanding it to walk into a wall effectively absorbs those slips and prevents accidental falls into adjacent holes, ultimately maximizing its chances of safely reaching the goal.

The value function represents the expected long-term, cumulative discounted reward for starting in a specific tile and following the policy perfectly until the game ends.

In [ ]:
# Run experiment with significantly lowered discount factor
exp_policy, exp_V, exp_iters = policy_iteration(env, gamma=0.1)

print_results(exp_policy, exp_V, "EXPERIMENT (Gamma = 0.10)")


EXPERIMENT (Gamma = 0.10)
Value Function:
[[0.    0.    0.    0.   ]
 [0.    0.    0.    0.   ]
 [0.    0.001 0.012 0.   ]
 [0.    0.012 0.345 0.   ]]

Policy:
[['↓' '↑' '→' '↑']
 ['←' 'H' '←' 'H']
 ['↑' '↓' '←' 'H']
 ['H' '→' '↓' 'G']]


A discount factor of 0.10 means that every step the agent takes reduces the perceived value of the final reward by a massive 90%. By the time the algorithm tries to propagate the expected value backward from the goal (bottom-right) to the starting position (top-left), the discounted reward decays to a number so small that it registers as 0.0. Since all possible actions from the start tile lead to an expected future value of exactly zero, the algorithm becomes completely apathetic; it acts arbitrarily in states far from the goal because it is too short-sighted to realize a distant reward even exists.

The Frozen Lake environment satisfies the Markov Property because the state transitions $P(s' \mid s, a)$ and rewards depend only on the current state and the chosen action, not on the agent's previous history. The stochastic nature of the environment (the `is_slippery` mechanic means an action 'Up' might result in moving 'Left' or 'Right' with some probability) requires the agent to plan for expected outcomes, making it a perfect discrete MDP.

The final policy (with $\gamma = 0.99$) maps every possible state to the optimal action that maximizes the expected cumulative discounted reward. Since the ice is slippery, the policy does not directly point straight to the goal. In states adjacent to holes, the policy often commands the agent to walk into a safe wall. Since the agent might slip perpendicular to its chosen direction, walking into a wall prevents it from accidentally slipping into a hole, acting as a calculated safety mechanism.

# Task 2

In this task, I will explore parameter-efficient fine-tuning (PEFT). To ground this exploration of PEFT, and specifically low-rank adaptation (LoRA), this analysis draws upon two foundational technical sources. The primary source is the seminal research paper introducing the method, *LoRA: Low-Rank Adaptation of Large Language Models* by Hu et al. (2021), which provides the theoretical mathematical framework for the architecture. To bridge this theory with modern practical implementation, the secondary source utilized is the official Hugging Face technical documentation, *PEFT: State-of-the-art Parameter-Efficient Fine-Tuning methods* (2023). Together, these sources illuminate one of the most critical breakthroughs in modern artificial intelligence. As large language models continue to scale to hundreds of billions, and even trillions, of parameters, adapting them to specialized tasks using traditional "full" fine-tuning has become prohibitively expensive, requiring massive clusters of high-end GPUs. Parameter-efficient fine-tuning methods solve this fundamental bottleneck by offering a way to achieve state-of-the-art task adaptation while modifying only a microscopic fraction of the model's overall architecture. Among these techniques, LoRA has rapidly emerged as the industry standard due to its elegant mathematical approach to weight updates.

The primary problem that PEFT methods attempt to solve is the astronomical computational cost of adapting modern large language models to specific tasks. When a foundation model contains tens or hundreds of billions of parameters, performing standard "full" fine-tuning, where every single weight is updated, requires massive clusters of GPUs just to hold the optimizer states and gradients in memory. Low-rank adaptation (LoRA) addresses this bottleneck through a clever mathematical insight: it operates on the hypothesis that the specific weight changes required to learn a new task have a very low "intrinsic dimension." The core idea is that it is not necessary to alter the entire massive neural network to teach it something new; it is more than sufficient to make targeted, low-dimensional adjustments. Architecturally, LoRA achieves this by completely freezing the original pre-trained weight matrices of the model. Instead of updating the massive original matrix $W$, LoRA injects two much smaller, trainable rank-decomposition matrices, typically denoted as $A$ and $B$, into the transformer layers. During a forward pass, the input is processed by both the frozen original weights and these new tiny matrices, and the results are added together. The training objective remains exactly the same as standard fine-tuning, such as cross-entropy loss for next-token prediction, but during backpropagation, the gradients are only calculated and applied to the microscopic $A$ and $B$ matrices. This dramatically slashes the memory and compute requirements while allowing the model to achieve performance that rivals or matches full fine-tuning.

To demonstrate the mechanical advantage of parameter-efficient fine-tuning, I will inspect exactly what happens to a foundation model's architecture when a LoRA adapter is injected. As an experiment, I will load a standard pre-trained GPT-2 model and apply a LoRA configuration with a rank ($r$) of 8. I will then programmatically count the total number of parameters in the model versus the number of parameters that actually require gradient updates (training) during backpropagation.

In [4]:
from transformers.utils import logging
from transformers import AutoModelForCausalLM
from peft import get_peft_model, LoraConfig, TaskType
import pandas as pd

logging.disable_progress_bar()

# Load the standard pre-trained foundation model
model_id = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_id)

# Define the LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

# Apply the LoRA adapters to the frozen model
peft_model = get_peft_model(model, lora_config)

# Extract parameter counts
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in peft_model.parameters())

# Generate output table
data = {
    "Metric": ["Total Parameters", "Trainable Parameters (LoRA)", "Trainable Parameters %"],
    "Value": [f"{all_params:,}", f"{trainable_params:,}", f"{(trainable_params / all_params) * 100:.3f}%"]
}
df = pd.DataFrame(data)

print("Table 1: Parameter Reduction using LoRA on GPT-2")
print("-" * 50)
print(df.to_string(index=False))

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Table 1: Parameter Reduction using LoRA on GPT-2
--------------------------------------------------
                     Metric       Value
           Total Parameters 124,734,720
Trainable Parameters (LoRA)     294,912
     Trainable Parameters %      0.236%


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


As demonstrated in the output, Table 1, the core theoretical claim of the LoRA architecture is practically validated. By utilizing a low-rank decomposition ($r=8$), approximately 124 million original parameters of the GPT-2 model are completely frozen. The only weights exposed to gradient updates are the newly injected $A$ and $B$ matrices, which total roughly 295,000 parameters. This suggests that actively training only $0.236\%$ of the network is sufficient. Since optimizer states (like Adam moments) and gradients only need to be stored for this minuscule fraction of the model, the VRAM required to fine-tune this model is drastically slashed, removing the hardware bottleneck while preserving the foundational knowledge of the original weights.

While LoRA provides massive computational benefits, it is not without limitations. First, it can struggle with "catastrophic forgetting" or limited expressiveness if the fine-tuning task requires a massive foundational shift in the model's knowledge base. Due to LoRA only optimizing a tiny, low-rank subspace of the network, it is highly effective for tasks like style transfer or behavioral alignment (such as instruction tuning), but it may fail to internalize completely new languages or complex, unseen domain knowledge as effectively as full fine-tuning. Second, the method introduces hyperparameter sensitivity that can offset some of its time-saving benefits. With LoRA, it is immensely important to carefully select the rank dimension ($r$) and the scaling factor ($\alpha$). If the rank is set too low, the adapter will lack the capacity to learn the task, resulting in underfitting. If set too high, the computational savings diminish, and the model risks overfitting to the training data. Finding the optimal configuration often requires empirical trial and error.

To build upon this structural analysis of parameter reduction, a concrete follow-up experiment would be to directly quantify the real-world trade-off between VRAM savings and task performance. I would propose taking the base GPT-2 model and training it on a standard sequence classification benchmark, such as the GLUE or IMDB sentiment datasets, using two parallel pipelines. Pipeline A would utilize standard full fine-tuning, while Pipeline B would utilize LoRA with a rank of 8. By logging the peak GPU memory utilization during training, the total training time, and the final validation accuracy of both models, it can be empirically determined whether the 99.7% reduction in trainable parameters leads to any statistically significant degradation in the model's predictive capabilities on a downstream task.